<center><h1><strong><font color="blue"> Data Engineering for Data Science - Week 06</font></strong></h1></center>

<center><img alt="" src="images/cover.png" style="height: 600px;"/></center>

<center><h2><strong><font color="blue">Advanced Selection, Aggregations, & Normalization</font></strong></h2></center>

<b><center><h3>(C) Taufik Sutanto</h3></center>

<center><h2><strong><font color="blue">Outline</font></strong></h2></center>

**Topics**:

* DDL (Create, Alter, Drop) vs. DML (Select, Insert, Update, Delete).
* Advanced Selection: Joins (Inner, Left, Right, Full) and Aggregations.
* The importance of data redundancy and anomalies in research data.
* Functional Dependencies.
* Normal Forms: 1NF, 2NF, and 3NF.
* Denormalization: When and why to break the rules.

  
**Lab**: 
* Given a flat, messy "Excel-style" dataset, decompose it into a 3NF schema.


<center><h2><strong><font color="blue"> Export-Import Data using phpMyAdmin</font></strong></h2></center>

<center><h3><strong><font color="red"> Please clone (“pull”) the course GitHub repository and locate the file "example-schema-sample-data.sql" within the "data" folder</font></strong></h3></center>

<center><img alt="" src="images/deds-04/phpmyadmin-eximport.jpg" style="height: 480px;"/></center>

<center><h2><strong><font color="red">Is this the only method for exporting and importing data to and from MySQL?</font></strong></h2></center>

<center><img alt="" src="images/curious.jpg" style="height: 320px;"/></center>

<center><img alt="" src="images/deds-05/university-registry-db-ERD.jpg" style="height: 480px;"/></center>

In [ ]:
# Best Practice; use a function to connect to our database
import warnings; warnings.simplefilter('ignore')
import mysql.connector
import pandas as pd

host = "localhost"
user = "root"
password = ""
database = "university_registry_db"

def connect(host="localhost", user="root", password="", database=""):
    try:
        db = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database)
        con = db.cursor()
        print("Connected to the '{}' Database.".format(database))
        return db, con
    except Exception as err_:
        print("Connection to the Database server failed:")
        print(err_)

## Data Manipulation Language (DML) in MariaDB
While Data Definition Language (DDL) builds the "house," **Data Manipulation Language (DML)** is the act of moving furniture in. It allows researchers to interact with the actual data stored within the tables. In the context of the **CRUD** (Create, Read, Update, Delete) cycle, DML is where the majority of your daily data engineering work occurs.

## 1. The SELECT Statement (Reading Data)

The `SELECT` statement is the most frequent operation in data science. It retrieves specific records from one or more tables based on defined criteria.

### Basic Syntax
```sql
SELECT id, full_name 
FROM students 
WHERE country_origin = 'Indonesia' 
ORDER BY full_name ASC;
```

* **Pro Tip:** Avoid `SELECT *` in production code. Explicitly naming columns reduces memory overhead and prevents your Python scripts from breaking if the table schema changes later.

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

qry = """ SELECT id, full_name
FROM students 
WHERE country_origin = 'Indonesia' 
ORDER BY full_name ASC;
"""
df = pd.read_sql(qry, db) # pay attention we are using db, and not "con". Pandas recommend using alchemy SQL connection
# Also pay attention we can only use pandas "read_sql" to get some data from the database. 
# We can not execute any arbitrary query that does not get data (e.g. update or delete data)
df.head()

con.close(); db.close() #Best practice to always close the connection regularly

<center><h2><strong><font color="red">Is the query case-sensitive? Is this behavior consistent across all cases?</font></strong></h2></center>

<center><img alt="" src="images/curious.jpg" style="height: 320px;"/></center>

## 2. The INSERT Statement (Creating Data)

`INSERT` adds new rows to a table. For production-grade research, we often handle this through Python scripts to ensure data is sanitized.

### Standard Insertion
```sql
INSERT INTO instructors (full_name, email) 
VALUES ('Dr. Aris Kusuma', 'aris.k@uiii.ac.id');
```

### Multi-row Insertion
```sql
INSERT INTO students (full_name, country_origin) 
VALUES 
    ('Siti Aminah', 'Indonesia'),
    ('John Doe', 'Australia'),
    ('Li Wei', 'China');
```

In [ ]:
# Good practice insert data ~ fast
qry = "INSERT INTO students (full_name, country_origin) VALUES (%s, %s)"
data = [('Siti Aminah', 'Indonesia'), ('John Doe', 'Australia'), ('Li Wei', 'China')]
data

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

con.executemany(qry, data)
db.commit() # Mandatory for data modification
print(f"Record inserted : {con.rowcount}") # you can also check from phpmyadmin

con.close(); db.close() #Best practice to always close the connection regularly

<center><h2><strong><font color="green">Essential for production pipelines</font></strong></h2></center>

<center><img alt="" src="images/recommended.jpg" style="height: 320px;"/></center>

In [ ]:
import logging

db, con = connect(host=host, user=user, password=password, database=database)

try:
    con.executemany(qry, data)
    db.commit() # Mandatory for data modification
    print(f"Record inserted : {con.rowcount}")
except Exception as e:
    logging.error("Insert failed", exc_info=True)
    con.rollback()

con.close(); db.close() #Best practice to always close the connection regularly

## 3. The UPDATE Statement (Modifying Data)

The `UPDATE` statement modifies existing records. In a professional environment, this must be handled with extreme precision.

### Critical Usage
```sql
UPDATE enrollments 
SET grade = 3.85 
WHERE student_id = 101 AND course_id = 5;
```

> **Warning:** Never omit the `WHERE` clause unless you intended to change the value for every single row in the table. There is no "undo" button for a mass update in a basic configuration.

In [ ]:
# Example of updating data from python
data = [(3.9, 1, 1), (2.9, 1, 2), (1.9, 2, 1)]
qry = "UPDATE enrollments SET grade = %s WHERE student_id = %s AND course_id = %s"
qry

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

con.executemany(qry, data)
db.commit() #check on phpMyAdmin
print(f"Record affected : {con.rowcount}")

con.close(); db.close() #Best practice to always close the connection regularly

<center><h2><strong><font color="green">Essential Tips Insert-Update</font></strong></h2></center>

## Logic: _“update if exists, otherwise insert”_

<center><img alt="" src="images/recommended.jpg" style="height: 320px;"/></center>

```sql
INSERT INTO students (id, full_name, country_origin, enrollment_year)
VALUES (1, 'Alice', 'Indonesia', 2023)
ON DUPLICATE KEY UPDATE
    full_name = VALUES(full_name),
    country_origin = VALUES(country_origin),
    enrollment_year = VALUES(enrollment_year);
```
### Behavior:
* If id = 1 does not exist → INSERT
* If id = 1 already exists → UPDATE that row

In [ ]:
qry = """
INSERT INTO students (id, full_name, country_origin, enrollment_year)
VALUES (%s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    full_name = VALUES(full_name),
    country_origin = VALUES(country_origin),
    enrollment_year = VALUES(enrollment_year)
"""
data = (1, "Siti Nurhaliza", "Indonesia", 2023)
data

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

con.execute(qry, data)
db.commit()

print(f"Record affected : {con.rowcount}")
con.close(); db.close() #Best practice to always close the connection regularly

## 4. The DELETE Statement (Removing Data)

* The `DELETE` statement removes specific rows from a table. Unlike the `DROP` command (which removes the table itself), `DELETE` only removes the (row) data.
* Be very careful when using `DELETE`. Deleting data in MySQL from Python is deceptively simple, but in production it’s where data loss, locking, and performance issues most often occur. 

### Targeted Deletion
```sql
DELETE FROM students WHERE id = 1
```

* **Primary key**: It is recommended to use primary key to delete data.

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

con.executemany(
    "DELETE FROM students WHERE id = %s",
    [(1,), (2,), (3,)])
db.commit()

print(f"Record affected : {con.rowcount}")
con.close(); db.close() #Best practice to always close the connection regularly

<center><h2><strong><font color="red">NEVER Run Unbounded DELETE (Critical)</font></strong></h2></center>

<center><img alt="" src="images/warning.jpg" style="height: 320px;"/></center>

<center><h3><strong><font color="green">e.g. "DELETE FROM students;"</font></strong></h3></center>

## DML Command Lifecycle

| Command | CRUD Equivalent | Research Utility |
| :--- | :--- | :--- |
| **SELECT** | Read | Extracting features for ML models or EDA. |
| **INSERT** | Create | Ingesting sensor data or web-scraped results. |
| **UPDATE** | Update | Cleaning "dirty" data or updating model logs. |
| **DELETE** | Delete | Removing outliers or outdated experimental runs. |

<center><h2><strong><font color="blue">Understanding MariaDB Character Set and Collation</font></strong></h2></center>

In database management, these two terms are often grouped together, but they handle very different parts of how your data is processed. Think of it as the difference between **knowing which letters exist** and **knowing how to alphabetize them**.

## 1. Character Set (The "What")
A **Character Set** is a defined list of characters and their corresponding binary values. It determines **what** specific symbols, letters, and numbers the database is capable of storing.

* **Function:** It maps a character (like 'A' or '😊') to a specific number (like 65 or a 4-byte Unicode sequence) that the computer understands.
* **Common Example:** `utf8mb4`. This set is widely used because it can store almost every character from every language, plus emojis and mathematical symbols.

## 2. Collation (The "How")
A **Collation** is a set of rules used to **compare and sort** the characters within a specific character set. It determines the logic used in `ORDER BY` clauses or when checking equality in a `WHERE` clause.

* **Function:** It defines whether the database should be case-sensitive (is 'A' the same as 'a'?) and how to handle accents (is 'e' the same as 'é'?).
* **Common Example:** `utf8mb4_unicode_ci`. The `_ci` at the end stands for **Case Insensitive**, meaning the database treats 'Apple' and 'apple' as identical when searching or sorting.

## 3. Key Differences at a Glance

| Feature | Character Set | Collation |
| :--- | :--- | :--- |
| **Primary Goal** | Storage and encoding. | Sorting and comparison. |
| **Responsibility** | Defines *which* characters are valid. | Defines *how* valid characters relate to each other. |
| **Analogy** | The symbols in an alphabet. | The rules in a dictionary for sorting words. |
| **Scope** | Determines disk space (e.g., 1-byte vs 4-byte). | Determines query results (e.g., sorting order). |

## 4. Why the Distinction Matters
If you choose the wrong **Character Set**, you might end up with "broken" characters (often appearing as `?` or ``) because the database doesn't recognize the symbol you're trying to save. 

If you choose the wrong **Collation**, your data will be stored correctly, but your search results might be unexpected—for instance, a search for "resume" might fail to find "Résumé," or your alphabetical list might put lowercase 'z' before uppercase 'A'.

<center><h2><strong><font color="blue">Understanding MariaDB Character Set and Collation</font></strong></h2></center>

<center><img alt="" src="images/deds-04/DB-collations-character-sets.png" style="height: 600px;"/></center>

### Related Queries

```sql
CREATE DATABASE research_db 
CHARACTER SET utf8mb4 
COLLATE utf8mb4_unicode_ci;
-----------------------------------
CREATE TABLE international_students (
    id INT PRIMARY KEY,
    student_name VARCHAR(100)
) CHARACTER SET utf8mb4 COLLATE utf8mb4_bin;
-----------------------------------
SHOW TABLE STATUS LIKE 'students';
```

<center><h2><strong><font color="blue">The MariaDB my.ini Configuration File</font></strong></h2></center>

<center><img alt="" src="images/deds-04/mysql-my-ini-file.png" style="height: 600px;"/></center>

<center><h1><strong><font color="blue">Advanced Selection: Joins (Inner, Left, Right, Full)</font></strong></h1></center>

# Advanced Selection: Inner Joins in Relational Databases

The following analysis examines the **Inner Join** operation, a critical component of relational algebra used to synthesize information across disparate tables. This operation facilitates the reconstruction of logical entities from normalized schemas.

## 1. Theoretical Overview of Inner Joins

In relational database management systems (RDBMS), an **Inner Join** creates a new result set by combining rows from two tables based on a common attribute. This operation strictly retrieves records where a match exists in both participants of the join.

### 1.1 Logical Mechanism
If $A$ and $B$ are two relations, the Inner Join ($A \Join B$) identifies the intersection based on a join predicate. If a row in table $A$ does not have a corresponding match in table $B$ (or vice versa), that row is excluded from the final output.

## 2. Case Study: `university_registry_db`

To illustrate this concept, we utilize the provided university schema containing the `instructors` and `courses` relations.

### 2.1 Schema Reference
* **Table `instructors`**: Primary key `id`, contains staff names and departments.
* **Table `courses`**: Primary key `id`, contains a foreign key `instructor_id` which references the instructor table.

### 2.2 Example Problem
**Objective:** Generate a report listing all course titles and the full names of the instructors assigned to them.

**SQL Logic:**
```sql
SELECT courses.title, instructors.full_name
FROM courses
INNER JOIN instructors ON courses.instructor_id = instructors.id;
```
In this query, the `ON` clause specifies the join condition: the `instructor_id` in the `courses` table must match the `id` in the `instructors` table.

<center><img alt="" src="images/deds-05/inner-join-db.png" style="height: 600px;"/></center>

## 3. Python Implementation
Data engineers frequently execute join operations programmatically to load relational data into analysis structures like **Pandas DataFrames**.

In [ ]:
# Define the Inner Join query
query = """
SELECT 
    c.title AS Course_Title, 
    i.full_name AS Instructor_Name,
    i.department AS Department
FROM courses c
INNER JOIN instructors i ON c.instructor_id = i.id;
"""

db, con = connect(host=host, user=user, password=password, database=database)

# Execute and load into a DataFrame
df_joined = pd.read_sql(query, db)
# Display the synthesized research data
print("Synthesized Course-Instructor List:")
print(df_joined.head())

# Close connection
con.close(); db.close() #Best practice to always close the connection regularly

### 3.1 Key Implementation Observations
* **Aliasing**: The use of `c` and `i` as table aliases improves code readability and reduces verbosity in complex joins.
* **Data Integrity**: Because `instructor_id` is a foreign key, the Inner Join ensures that the report only includes courses with valid, assigned academic staff.

## 4. Summary inner join
The Inner Join is the standard method for querying related data in a normalized database. It prioritizes the **intersection** of datasets, ensuring that only complete records are processed.

# Advanced Selection: The Left Join Operation

The **Left Join** (or Left Outer Join) is a pivotal operation in relational algebra used to combine data from two tables while preserving the integrity of the primary (left) dataset. Unlike the Inner Join, which only returns matching rows, the Left Join ensures that no information from the leading table is lost during the query process.

## 1. Theoretical Framework
In a relational database, a Left Join returns all records from the "left" table and the matched records from the "right" table. If the join condition finds no match in the right table, the resulting columns for that table are populated with **NULL** values.

### 1.1 Logical Rationale
* **Data Preservation**: It is used when the researcher needs a complete list from the primary table regardless of whether associated data exists in the secondary table.
* **Gap Analysis**: This join is essential for identifying "missing" relationships, such as students who have not yet enrolled in any courses or instructors who are not yet assigned to a class.

## 2. Case Study: Identifying Faculty Assignments
Using the `university_registry_db` schema, we examine the relationship between the `instructors` and `courses` tables. 

### 2.1 The Problem
**Objective**: Generate a comprehensive report of all academic staff. The report must include the titles of the courses they teach. If an instructor is not currently teaching, their name must still appear in the list with a "NULL" or empty course designation.

**SQL Logic**:
```sql
SELECT 
    i.full_name AS instructor_name, 
    c.title AS course_title
FROM instructors i
LEFT JOIN courses c ON i.id = c.instructor_id;
```

<center><img alt="" src="images/deds-05/left-join-db.png" style="height: 600px;"/></center>

## 3. Pythonic Execution

Data scientists often utilize the `pandas` library to perform these operations during the data cleaning and preparation phases of research.

In [ ]:
query = """
SELECT 
    i.full_name, 
    i.department, 
    c.title AS assigned_course
FROM instructors i
LEFT JOIN courses c ON i.id = c.instructor_id;
"""

db, con = connect(host=host, user=user, password=password, database=database)

# Load the result set into a DataFrame
df_faculty = pd.read_sql(query, db)

# Identify instructors without assignments
# These will have NaN values in the 'assigned_course' column
unassigned = df_faculty[df_faculty['assigned_course'].isna()]

print("Comprehensive Faculty Report:")
print(df_faculty.head(10))

# Close the database resources
con.close(); db.close() #Best practice to always close the connection regularly

## 4. Key Distinctions

The Left Join is characterized by its asymmetric behavior:
* **Left Table**: Every row is retained.
* **Right Table**: Only matching rows are retained; others become NULL.
* **Usage**: Primary for reporting and auditing where "missing" data is as informative as "present" data.

This approach is vital for maintaining a "Single Source of Truth" in complex research environments where datasets are frequently updated but must remain consistent across tables.

# Advanced Selection: The Right Join Operation

In the realm of relational algebra, the **Right Join** (or Right Outer Join) serves as the logical mirror to the Left Join. While less frequently used in daily practice—mostly because developers tend to think "Left-to-Right"—it is a vital tool for ensuring the integrity of secondary datasets.

## 1. Theoretical Overview
A **Right Join** retrieves all records from the "Right" table (Table B) and the matched records from the "Left" table (Table A). If there is no match for a specific row in the right table within the left table, the resulting columns for the left table will be populated with **NULL** values.

### 1.1 Why use a Right Join?
* **Completeness of the Secondary Set**: Use this when your primary interest lies in the records of the second table mentioned in your query.
* **Orphan Detection**: It is particularly useful for finding "orphaned" records—rows in a child table that do not have a corresponding parent in the main table.
* **Refining Assignments**: In academic databases, it helps identify courses that have been created but have not yet been assigned to an instructor.

## 2. Case Study: Auditing the `university_registry_db`
Let's look at our university schema, specifically the relationship between `instructors` and `courses`.

### 2.1 The Example Problem
**Scenario**: You are performing an administrative audit. You need a list of **all courses** in the catalog, along with the names of the instructors assigned to them. Crucially, you must include courses that currently have **no assigned instructor** (where `instructor_id` might be `NULL` or point to a deleted record).

**SQL Logic**:
```sql
SELECT 
    i.full_name AS Instructor_Name, 
    c.title AS Course_Title
FROM instructors i
RIGHT JOIN courses c ON i.id = c.instructor_id;
```

In this query, `courses` is the "Right" table. Every single course will appear in the result set. If a course like "Advanced Quantum Ethics" exists but has no instructor, the `Instructor_Name` column will simply show `NULL`.

<center><img alt="" src="images/deds-05/right-join-db.png" style="height: 600px;"/></center>

## 3. Python Implementation
When working in a research pipeline, you’ll likely use **Pandas** to handle the results of your SQL joins for further analysis.

In [ ]:
# The Right Join Query
# We want every course, regardless of instructor status
query = """
SELECT 
    i.full_name AS instructor, 
    c.title AS course,
    c.credits
FROM instructors i
RIGHT JOIN courses c ON i.id = c.instructor_id;
"""

db, con = connect(host=host, user=user, password=password, database=database)
# Execute and load into DataFrame
df_audit = pd.read_sql(query, db)
# Pro-tip: Identify 'TBD' courses instantly
tbd_courses = df_audit[df_audit['instructor'].isna()]

print("Full Course Audit (including unassigned):")
print(df_audit.head(10))
print("\nCourses requiring an instructor assignment:")
print(tbd_courses)

con.close(); db.close() #Best practice to always close the connection regularly

## 4. The "Mirror" Reality: Right vs. Left

It is worth noting that `A RIGHT JOIN B` is functionally identical to `B LEFT JOIN A`. Most data engineers prefer to use `LEFT JOIN` exclusively and simply reorder their tables to maintain a consistent mental flow. However, understanding the Right Join is essential for reading legacy code or passing those pesky technical certification exams.

# Advanced Selection: Full Outer Joins

The **Full Outer Join** represents the most comprehensive join operation in relational algebra. It serves as a set-theoretic union of the Left and Right joins, ensuring that no record from either participating relation is omitted from the resulting dataset.

## 1. Theoretical Framework
A **Full Join** retrieves all records from both the "Left" table (Table A) and the "Right" table (Table B). When a match exists based on the join predicate, the rows are combined. In instances where no match occurs, the result set still includes the unmatched row, populating the columns of the corresponding table with **NULL** values.

<center><img alt="" src="images/deds-05/full-join-db.png" style="height: 600px;"/></center>

### 1.1 Structural Properties
* **Total Retention**: Every entity from both relations is preserved in the output.
* **Bidirectional Nullability**: NULL values may appear in columns originating from either the left or right table, depending on which side lacks a matching pair.
* **Database Compatibility**: It is critical to note that MariaDB and MySQL do not natively support the `FULL OUTER JOIN` syntax. Execution requires a `UNION` operation between a `LEFT JOIN` and a `RIGHT JOIN`.

## 2. Case Study: Comprehensive Institutional Auditing

Utilizing the `university_registry_db` schema, we examine the `instructors` and `courses` relations.

### 2.1 The Objective
Construct a master registry that displays every instructor and every course currently in the database. This report must highlight:
1.  Instructors who are not currently assigned to any courses.
2.  Courses that have not yet been assigned to an instructor.

### 2.2 Relational Logic (MariaDB Workaround)
```sql
SELECT i.full_name, c.title
FROM instructors i
LEFT JOIN courses c ON i.id = c.instructor_id
UNION
SELECT i.full_name, c.title
FROM instructors i
RIGHT JOIN courses c ON i.id = c.instructor_id;
```
This query ensures that both "unassigned instructors" and "unassigned courses" are captured in a single unified view.

## 3. Python Implementation

For professional data engineering workflows, the `pandas` library offers a streamlined `merge` function that natively supports the "outer" join logic.

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

# Fetching individual tables to perform the join in Python
# Alternatively, execute the UNION SQL query directly
instructors_df = pd.read_sql("SELECT id, full_name, department FROM instructors", db)
courses_df = pd.read_sql("SELECT title, instructor_id FROM courses", db)

# Execute Full Outer Join using Pandas
# 'how="outer"' handles the union logic automatically
full_registry = pd.merge(
    instructors_df, 
    courses_df, 
    left_on='id', 
    right_on='instructor_id', 
    how='outer')

# Identify incomplete records for administrative review
missing_assignments = full_registry[
    full_registry['full_name'].isna() | full_registry['title'].isna()]

print("Master Institutional Registry (Full Join Result):")
print(full_registry.head(15))

print("\nRecords requiring immediate attention (Unmatched):")
print(missing_assignments)

con.close(); db.close() #Best practice to always close the connection regularly

## 4. Evaluation of Utility

The Full Join is indispensable for high-integrity data auditing. While the **Inner Join** prioritizes precision and **Outer Joins** prioritize a specific direction, the **Full Join** prioritizes the **totality** of the research environment. This approach is fundamental for detecting data entry gaps or structural inconsistencies in complex academic systems.

Does this comprehensive view clarify how relational logic maintains institutional consistency?

<center><h1><strong><font color="blue">Database Aggregations</font></strong></h1></center>

In the architecture of data engineering, **Aggregation** is the process of distilling large volumes of atomic records into meaningful summary statistics. As a lecturer in Data Science, you likely recognize this as the fundamental "reduction" step required before data can be visualized or modeled.

## 1. The Core Aggregate Functions
SQL provides a suite of mathematical functions designed to operate on a set of values and return a single scalar result. These functions are typically used in conjunction with the `GROUP BY` clause.

| Function | Purpose | Case Study Application |
| :--- | :--- | :--- |
| `COUNT()` | Returns the number of rows. | Number of students enrolled per course. |
| `SUM()` | Calculates the total of a numeric column. | Total credits a student is currently taking. |
| `AVG()` | Calculates the arithmetic mean. | Determining the GPA of a specific student or class average. |
| `MIN()` | Identifies the lowest value. | Finding the lowest grade in a difficult course. |
| `MAX()` | Identifies the highest value. | Finding the top performer in a research cohort. |

## 2. The `GROUP BY` and `HAVING` Logic
Aggregation is rarely performed on the entire table. Instead, we "bucket" the data using `GROUP BY`.

### 2.1 The Syntax of the Pivot
When you aggregate, every column in your `SELECT` statement must either be an **aggregate function** or part of the **`GROUP BY`** clause.

### 2.2 Filtering Aggregates: `HAVING` vs `WHERE`
* **`WHERE`**: Filters individual rows *before* they are grouped.
* **`HAVING`**: Filters the groups *after* the aggregation is calculated.

> **Research Note:** Use `WHERE` to exclude outliers (e.g., test runs with sensor errors) and `HAVING` to filter results (e.g., only showing courses with more than 10 students).

<center><img alt="" src="images/deds-05/database-aggregations.png" style="height: 600px;"/></center>

## 3. Case Study: Analytics on `university_registry_db`
Based on the provided schema, we will execute two specific research queries.

### Query A: Average Grade per Course
To evaluate course difficulty, we calculate the mean grade ($AVG = \frac{\sum Grade}{N}$) for every course title.

```sql
SELECT c.title, AVG(e.grade) as avg_grade
FROM courses c
JOIN enrollments e ON c.id = e.course_id
GROUP BY c.title;
```

### Query B: Student Diversity Index
Counting the number of students from each country of origin.

```sql
SELECT country_origin, COUNT(id) as student_count
FROM students
GROUP BY country_origin
ORDER BY student_count DESC;
```

## 4. Python Implementation

For professional data science workflows, loading aggregated SQL results directly into a **Pandas DataFrame** is the standard approach for downstream analysis.

In [ ]:
db, con = connect(host=host, user=user, password=password, database=database)

# Implementation of Query A: Grade Performance Analysis
query = """
SELECT 
    c.title AS Course, 
    ROUND(AVG(e.grade), 2) AS Mean_Grade,
    COUNT(e.student_id) AS Enrollment_Count
FROM courses c
JOIN enrollments e ON c.id = e.course_id
GROUP BY c.title
HAVING Enrollment_Count > 1
ORDER BY Mean_Grade DESC;
"""

# Load and visualize
df_performance = pd.read_sql(query, db)

print("Course Performance Metrics:")
print(df_performance)

# Close resource
con.close(); db.close() #Best practice to always close the connection regularly

## 5. Summary

Aggregation transforms a database from a mere storage locker into a primary analysis tool. By performing these calculations on the **database server** rather than in Python memory, you minimize data transfer and leverage the optimized indexing of the MariaDB engine.

<center><h2><strong><font color="blue">Prologue/Motivation ~ Normalization</font></strong></h2></center>

<center><img alt="" src="images/deds-05/motivation-normalization.png" style="height: 600px;"/></center>

# Database Design: Redundancy, Anomalies, and Functional Dependencies

Efficient database design is critical for maintaining data integrity, especially in research environments where data accuracy and consistency are paramount. This explanation covers the pitfalls of poor schema design and the formal logic used to correct them.

## 1. Data Redundancy and Anomalies in Research Data

In a relational database, **data redundancy** occurs when the same piece of data is stored in multiple places unnecessarily. While redundant backups are good, redundancy within a schema leads to significant operational risks known as **anomalies**.

### 1.1 The Impact of Redundancy
* **Storage Inefficiency:** Large-scale research datasets (e.g., genomic data or longitudinal surveys) can become prohibitively expensive to store if duplicate information is recorded across millions of rows.
* **Data Inconsistency:** The primary danger is that one instance of a data point may be updated while others are not, leading to conflicting "truths" within the same dataset.

### 1.2 Database Anomalies
Anomalies are problems that arise during the manipulation of a poorly structured table. They are generally categorized into three types:

| Anomaly Type | Description | Research Example |
| :--- | :--- | :--- |
| **Update Anomaly** | Occurs when a change to a single data value requires multiple rows to be updated. | If a Lab Location changes, and it is stored in every row of an "Experiments" table, failing to update every row leads to inconsistent records. |
| **Insertion Anomaly** | Occurs when certain data cannot be recorded unless other, unrelated data is also provided. | You cannot record a new Researcher's profile until they are assigned to an active Experiment, because the primary key requires an Experiment ID. |
| **Deletion Anomaly** | Occurs when deleting one piece of information unintentionally results in the loss of other unrelated data. | Deleting the last Experiment entry for a specific Lab might accidentally delete all contact information for that Lab. |




## 2. Functional Dependencies (FD)

**Functional Dependency** is a constraint between two sets of attributes in a relation. It is the fundamental building block of **Normalization**, the process used to eliminate redundancy and anomalies.

### 2.1 Formal Definition
If $R$ is a relation and $X$ and $Y$ are subsets of attributes in $R$, we say $Y$ is **functionally dependent** on $X$ (denoted as $X \to Y$) if and only if each value of $X$ is associated with exactly one value of $Y$.

In simpler terms, if you know the value of $X$, you can determine the value of $Y$.

> **Example:** In a research database, `ResearcherID \to ResearcherName`. 
> If `ResearcherID` is 101, the name will always be "Dr. Smith." You won't find `ResearcherID` 101 associated with "Dr. Jones."



### 2.2 Types of Functional Dependencies

To refine database structures, we identify specific categories of FDs:

#### A. Full Functional Dependency
An attribute is fully functionally dependent if it depends on the *entire* primary key, not just a part of it.
* **Example:** In a table where the key is `{Student_ID, Course_ID}`, the `Grade` is fully dependent because you need both the student and the course to determine the grade.

#### B. Partial Functional Dependency
This occurs when an attribute depends on only a portion of a composite primary key. This is a primary cause of redundancy.
* **Example:** In the same `{Student_ID, Course_ID}` table, `Student_Name` is only dependent on `Student_ID`. Storing it here creates partial dependency.

#### C. Transitive Dependency
This occurs when a non-key attribute depends on another non-key attribute, which in turn depends on the primary key ($A \to B$ and $B \to C$).
* **Example:** `Project_ID \to Department_ID` and `Department_ID \to Department_Head`. Here, `Department_Head` has a transitive dependency on `Project_ID`.

### 2.3 Role in Normalization
Functional dependencies allow us to decompose a single, bloated table into smaller, logically sound tables:
1.  **First Normal Form (1NF):** Remove multi-valued attributes.
2.  **Second Normal Form (2NF):** Remove **Partial Dependencies**.
3.  **Third Normal Form (3NF):** Remove **Transitive Dependencies**.

By identifying $X \to Y$ relationships, researchers ensure that each fact is stored exactly once, preserving the integrity of the scientific record.

<center><img alt="" src="images/deds-05/functional-dependency.png" style="height: 600px;"/></center>

# Second Normal Form (2NF)

We review the transition from First Normal Form (1NF) to **Second Normal Form (2NF)**. While 1NF ensures atomicity, it often retains structural flaws that lead to data redundancy. The objective of 2NF is to eliminate **Partial Functional Dependencies**.

## 1. Prerequisites and Initial State
To achieve 2NF, a relation must first satisfy all criteria for **1NF**. We begin with the following 1NF table from the `university_registry_db`, where the composite primary key is defined as `{StudentID, CourseID}`.

### Table: `Student_Course_Registry_1NF`
| StudentID | CourseID | StudentName | Major | CourseTitle | Grade |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 2021001 | MATH101 | Alice Smith | Mathematics | Algebra | A |
| 2021001 | PHYS202 | Alice Smith | Mathematics | Mechanics | B |
| 2021002 | CS101 | Bob Johnson | Data Science | Intro to CS | A |

> **Reviewer's Note:** Observe that the primary key is a composite of two attributes. Any attribute that depends on only one part of this key is considered a "partial dependency."

## 2. Identifying Partial Functional Dependencies

In 2NF, every non-prime attribute (attributes not part of any candidate key) must be **fully functionally dependent** on the entire primary key. We identify the following dependencies in our table:

1.  **Full Dependency:** `{StudentID, CourseID} $\to$ Grade` (A grade requires both the student and the specific course).
2.  **Partial Dependency A:** `StudentID $\to$ {StudentName, Major}` (The student's name does not change based on the course).
3.  **Partial Dependency B:** `CourseID $\to$ CourseTitle` (The course title is independent of which student is enrolled).

## 3. The Decomposition Methodology
The reviewer suggests a systematic decomposition to resolve these dependencies. We must separate the partially dependent attributes into new relations where they depend on a **whole** primary key.

### Step 1: Isolate Student Data
Create a `Students` table where `StudentID` is the unique primary key. This removes the need to repeat the student's name for every course they take.

### Step 2: Isolate Course Data
Create a `Courses` table where `CourseID` is the primary key. This prevents repeating the course title for every enrolled student.

### Step 3: Maintain Relationships
Retain a "link" or "junction" table to store attributes that truly depend on both keys (like the grade).

## 4. Resulting 2NF Schema

The original bloated table is now decomposed into three logically sound relations.

### Table: `Students`
| StudentID | StudentName | Major |
| :--- | :--- | :--- |
| 2021001 | Alice Smith | Mathematics |
| 2021002 | Bob Johnson | Data Science |

### Table: `Courses`
| CourseID | CourseTitle |
| :--- | :--- |
| MATH101 | Algebra |
| PHYS202 | Mechanics |
| CS101 | Intro to CS |

### Table: `Enrollments`
| StudentID | CourseID | Grade |
| :--- | :--- | :--- |
| 2021001 | MATH101 | A |
| 2021001 | PHYS202 | B |
| 2021002 | CS101 | A |

## 5. Summary of 2NF Compliance
A relation is in **Second Normal Form** if:
1.  It is already in **1NF**.
2.  It contains **no partial functional dependencies**.
3.  All non-key attributes are functionally dependent on the **entire** primary key.

By achieving 2NF, we have successfully mitigated the **Update Anomaly**. For instance, if a student changes their major, we only need to update a single row in the `Students` table.

<center><img alt="" src="images/deds-05/2NF-db.png" style="height: 600px;"/></center>

# Analysis of Third Normal Form (3NF)

The following discourse evaluates the transition from Second Normal Form (2NF) to **Third Normal Form (3NF)**. This phase of normalization is essential for eliminating **transitive functional dependencies**, ensuring that every non-key attribute depends strictly on the primary key.

## 1. Prerequisites and Initial State

To qualify for 3NF, a relation must first satisfy all criteria for **2NF**. We examine a single table from the `university_registry_db` that, while in 2NF, still harbors structural redundancies.

### Table: `Student_Department_2NF`

| StudentID (PK) | StudentName | DeptID | DeptName | DeptHead |
| :--- | :--- | :--- | :--- | :--- |
| 2021001 | Alice Smith | D01 | Mathematics | Dr. Aris |
| 2021002 | Bob Johnson | D02 | Data Science | Dr. Budi |
| 2021003 | Charlie Davis | D01 | Mathematics | Dr. Aris |

### Observations:
In this schema, `StudentID` is the primary key. However, we observe a multi-step dependency chain:
1.  **Direct Dependency:** `StudentID` determines `DeptID`.
2.  **Transitive Dependency:** `DeptID` determines `DeptName` and `DeptHead`.

Consequently, `DeptName` is transitively dependent on `StudentID` via `DeptID`. This creates a redundancy where department details are repeated for every student in that department.

## 2. The Decomposition Methodology

The reviewer suggests a systematic decomposition to remove the intermediate determinant. This process isolates the transitive attributes into a distinct relation.

### Step 1: Identify the Determinant
Identify the non-key attribute that acts as a determinant for other non-key attributes. In our case, `DeptID` determines `DeptName` and `DeptHead`.

### Step 2: Extract to a New Relation
Remove the transitively dependent columns (`DeptName`, `DeptHead`) from the original table and place them into a new `Departments` table. In this new table, `DeptID` serves as the primary key.

### Step 3: Establish a Logical Link
Retain `DeptID` in the original table. It now functions as a **foreign key**, linking the student record to the correct department data without duplicating descriptive text.

## 3. Resulting 3NF Schema
The decomposition results in two streamlined, logically sound tables.

### Table: `Students`
| StudentID (PK) | StudentName | DeptID (FK) |
| :--- | :--- | :--- |
| 2021001 | Alice Smith | D01 |
| 2021002 | Bob Johnson | D02 |
| 2021003 | Charlie Davis | D01 |

### Table: `Departments`
| DeptID (PK) | DeptName | DeptHead |
| :--- | :--- | :--- |
| D01 | Mathematics | Dr. Aris |
| D02 | Data Science | Dr. Budi |

## 4. Summary of 3NF Compliance

A relation is in **Third Normal Form** if:
1.  It satisfies all requirements of **2NF**.
2.  It contains **no transitive dependencies**.
3.  Every non-key attribute is dependent **only** on the primary key.

In academic terms, this state ensures that each attribute provides a fact about "the key, the whole key, and nothing but the key."

## 5. Normalization Beyond 3NF

While 3NF is typically sufficient for most research and enterprise applications, advanced normalization forms exist to handle more complex scenarios. These include **Boyce-Codd Normal Form (BCNF)** for overlapping candidate keys, as well as **4NF** and **5NF** for managing multi-valued and join dependencies.

<center><img alt="" src="images/deds-05/3NF-db.png" style="height: 600px;"/></center>

# Denormalization: Strategic Redundancy in Database Management

While normalization seeks to eliminate redundancy and preserve integrity, **denormalization** is a strategic architectural choice to reintroduce redundancy. This technique is employed when the computational cost of data retrieval outweighs the benefits of a normalized schema.

## 1. The Rationale: Why Break the Rules?

The reviewer notes that normalization, while theoretically sound, often imposes significant overhead during query execution. The primary motivations for denormalization include:

* **Performance Optimization:** In systems with high-frequency read operations, the cost of performing multiple $R_1 \Join R_2 \Join R_3$ (JOIN) operations can lead to unacceptable latency. Denormalization reduces the number of joins required.
* **Query Simplification:** Complex queries involving many tables are difficult to write and maintain. Merging related attributes into a single relation simplifies the application logic.
* **Reporting and Analytics:** In Online Analytical Processing (OLAP) environments, historical data is often denormalized into "Star Schemas" to facilitate rapid aggregation and trend analysis.

## 2. Triggering Conditions: When to Denormalize

Denormalization is not a license for poor design but a deliberate optimization. It should be considered under the following conditions:

1.  **High Read-to-Write Ratio:** When data is read thousands of times for every single update, the cost of redundancy (increased update time) is negligible compared to the read savings.
2.  **Strict Latency Requirements:** If a system must deliver responses in milliseconds and the JOIN overhead exceeds this threshold.
3.  **Derived Data Requirements:** When a specific value (e.g., a total sum or average) is frequently requested but expensive to calculate on the fly.
4.  
## 3. Case Study: The Research Publication Registry

Consider a normalized system tracking research papers and their departments.

**Normalized State (3NF):**
* `Papers` table (PaperID, Title, DeptID)
* `Departments` table (DeptID, DeptName)

To display a list of paper titles with their department names, the system must perform a JOIN for every request.

**Denormalized State:**
* `Papers_Denormalized` table (PaperID, Title, DeptID, **DeptName**)

By storing the `DeptName` directly in the `Papers` table, the JOIN is eliminated. The reviewer cautions, however, that if a department name changes, the system must now update multiple rows instead of one.

## 4. Implementation via Python

In modern data science workflows, denormalization is frequently performed during the data preparation phase using the `pandas` library.

```python
import pandas as pd

# 1. Normalized Data Sources
papers_df = pd.DataFrame({
    'PaperID': [101, 102, 103],
    'Title': ['Green AI', 'Big Data Ethics', 'EcoCluster'],
    'DeptID': ['D1', 'D1', 'D2']
})

depts_df = pd.DataFrame({
    'DeptID': ['D1', 'D2'],
    'DeptName': ['Computer Science', 'Environmental Science']
})

# 2. Denormalization Process (The "Break")
# We merge the tables to create a flat, redundant structure for faster access
denormalized_df = pd.merge(papers_df, depts_df, on='DeptID', how='left')

# 3. View the Resulting Redundancy
print("Denormalized Research Registry:")
print(denormalized_df)

# The 'DeptName' is now repeated, allowing for O(1) access per row 
# without further lookups or joins.
```

## 5. Summary Conclusion

Denormalization is a trade-off: you sacrifice **storage efficiency** and **write simplicity** to gain **read speed**. The reviewer suggests that for large-scale research infrastructures, such as those used in Green AI or high-frequency data streaming, denormalization is often a necessary departure from traditional relational theory.

<center><img alt="" src="images/deds-05/denormalization-db.png" style="height: 600px;"/></center>

<center><h2><strong><font color="green">Start designing database schema for your thesis project</font></strong></h2></center>

<center><img alt="" src="images/recommended.jpg" style="height: 320px;"/></center>

## Next Week's Discussions

* Conceptual vs. Logical vs. Physical Data Models.
* Entity-Relationship Diagrams (ERD).
* OLTP (Transactional) vs. OLAP (Analytical) systems.
* Introduction to Dimensional Modeling (Star Schema vs. Snowflake Schema).


<center><h2><strong><font color="blue">End of Module</font></strong></h2></center>

<center><img alt="" src="images/meme-cartoon/meme-db-joins.jpg" style="height: 480px;"/></center>